<a href="https://colab.research.google.com/github/brutal588/playing_pose_tracking/blob/main/docs/notebooks/Training_and_inference_on_an_example_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training and inference on an example dataset

In this notebook we'll install [`sleap-nn`](https://nn.sleap.ai/latest), download a sample dataset, run training and inference on that dataset using the `sleap-nn`, and then download the predictions.

**Note:** If you only need to perform training and inference (and not use the full SLEAP GUI or labeling tools), you don't need to install the entire `sleap` package. You can just install [`sleap-nn`](https://nn.sleap.ai/latest), which is a lighter-weight package focused on model training and inference.


## Install sleap-nn


In [2]:
# !pip install -qqq "sleap-nn[torch-cpu]"

# if you have GPU (in colab, enable GPU runtime)
!pip install -qqq "sleap-nn[torch-cuda128]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.8/691.8 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39

## Download sample training data into Colab
Let's download a sample dataset from the SLEAP [sample datasets repository](https://github.com/talmolab/sleap-datasets) into Colab.

In [3]:
!apt-get install tree
!wget -O dataset.zip https://github.com/talmolab/sleap-datasets/releases/download/dm-courtship-v1/drosophila-melanogaster-courtship.zip
!mkdir dataset
!unzip dataset.zip -d dataset
!rm dataset.zip
!tree dataset

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (977 kB/s)
Selecting previously unselected package tree.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
--2026-07-02 06:02:15--  https://github.com/talmolab/sleap-datasets/releases/download/dm-courtship-v1/drosophila-melanogaster-courtship.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting 

## Train models
For the top-down pipeline, we'll need train two models: a centroid model and a centered-instance model.

We'll first train a model for centroids using the default **training profile**. The training profile determines the model architecture, the learning rate, and other parameters.

When you start training, you'll first see the training parameters and then the training and validation loss for each training epoch.

As soon as you're satisfied with the validation loss you see for an epoch during training, you're welcome to stop training by clicking the stop button. The version of the model with the lowest validation loss is saved during training, and that's what will be used for inference.

If you don't stop training, it will run for 200 epochs or until validation loss fails to improve for some number of epochs (controlled by the `early_stopping` fields in the training profile).

In [4]:
# Let's get the default config files
!wget -O baseline.centroid.yaml https://raw.githubusercontent.com/talmolab/sleap-nn/refs/heads/main/docs/sample_configs/config_centroid_unet.yaml
!wget -O baseline.centered_instance.yaml https://raw.githubusercontent.com/talmolab/sleap-nn/refs/heads/main/docs/sample_configs/config_topdown_centered_instance_unet_medium_rf.yaml

--2026-07-02 06:02:23--  https://raw.githubusercontent.com/talmolab/sleap-nn/refs/heads/main/docs/sample_configs/config_centroid_unet.yaml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3513 (3.4K) [text/plain]
Saving to: ‘baseline.centroid.yaml’

baseline.centroid.y 100%[===================>]   3.43K  --.-KB/s    in 0s      

2026-07-02 06:02:23 (49.8 MB/s) - ‘baseline.centroid.yaml’ saved [3513/3513]

--2026-07-02 06:02:23--  https://raw.githubusercontent.com/talmolab/sleap-nn/refs/heads/main/docs/sample_configs/config_topdown_centered_instance_unet_medium_rf.yaml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.

In [6]:
!sleap-nn train --config baseline.centroid.yaml "data_config.train_labels_path=[dataset/drosophila-melanogaster-courtship/courtship_labels.slp]" trainer_config.ckpt_dir="models" trainer_config.run_name="courtship.centroid" trainer_config.max_epochs=200

2026-07-02 06:06:19 | Input config:
2026-07-02 06:06:19 | 
data_config:
  train_labels_path:
  - dataset/drosophila-melanogaster-courtship/courtship_labels.slp
  val_labels_path: null
  validation_fraction: 0.1
  use_same_data_for_val: false
  test_file_path: null
  provider: LabelsReader
  user_instances_only: true
  data_pipeline_fw: torch_dataset
  cache_img_path: null
  use_existing_imgs: false
  delete_cache_imgs_after_training: true
  preprocessing:
    ensure_rgb: false
    ensure_grayscale: false
    max_height: null
    max_width: null
    scale: 0.5
    crop_size: null
    min_crop_size: 100
  use_augmentations_train: true
  augmentation_config:
    intensity:
      uniform_noise_min: 0.0
      uniform_noise_max: 0.04
      uniform_noise_p: 0.0
      gaussian_noise_mean: 0.0
      gaussian_noise_std: 0.02
      gaussian_noise_p: 0.0
      contrast_min: 0.9
      contrast_max: 1.1
      contrast_p: 0.0
      brightness_min: 0.9
      brightness_max: 1.1
      brightness_p: 0.0

### Strategies to Improve GPU Utilization:

1.  **Increase Batch Size**: A larger batch size means the GPU processes more data in parallel, which can keep it busy and improve throughput. However, this is limited by GPU memory.
2.  **Increase Number of Data Loader Workers**: If the CPU is a bottleneck in preparing data for the GPU, increasing the number of `num_workers` can help. This allows multiple CPU processes to load and preprocess data concurrently.
3.  **Enable Mixed Precision Training**: Using `float16` (half-precision) instead of `float32` (single-precision) can significantly reduce memory usage and speed up computations on compatible GPUs, potentially allowing for even larger batch sizes or faster training.
4.  **Gradient Accumulation**: If a very large batch size is desired but doesn't fit into GPU memory, gradient accumulation can simulate a larger effective batch size by accumulating gradients over several smaller batches before performing an optimizer step. (This specific strategy might require direct modification of the `sleap-nn` configuration files or be a built-in option, so we'll focus on the more direct command-line parameters first.)

Let's apply some of these to the `sleap-nn` training command. We'll try increasing the `batch_size`, `num_workers`, and enabling mixed precision if the config supports it via `trainer_config.mixed_precision`.

In [7]:
# Original command:
# !sleap-nn train --config baseline.centroid.yaml "data_config.train_labels_path=[dataset/drosophila-melanogaster-courtship/courtship_labels.slp]" trainer_config.ckpt_dir="models" trainer_config.run_name="courtship.centroid" trainer_config.max_epochs=200

# Modified command to improve GPU utilization:
# Increasing batch_size and num_workers. Adding mixed_precision to '16-mixed' if supported.
# Using '+' prefix for trainer_config parameters as required by Hydra.
!sleap-nn train \
  --config baseline.centroid.yaml \
  "data_config.train_labels_path=[dataset/drosophila-melanogaster-courtship/courtship_labels.slp]" \
  trainer_config.ckpt_dir="models" \
  trainer_config.run_name="courtship.centroid_gpu_optimized" \
  trainer_config.max_epochs=200 \
  +trainer_config.batch_size=16 \
  +data_config.num_workers=8 \
  +trainer_config.mixed_precision=16-mixed

# Note: You might need to adjust batch_size and num_workers based on your specific GPU memory and CPU cores.

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/hydra/_internal/config_loader_impl.py", line 390, in _apply_overrides_to_config
    OmegaConf.update(cfg, key, value, merge=True)
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/omegaconf.py", line 741, in update
    root.__setattr__(last_key, value)
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/dictconfig.py", line 337, in __setattr__
    raise e
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/dictconfig.py", line 334, in __setattr__
    self.__set_impl(key, value)
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/dictconfig.py", line 318, in __set_impl
    self._set_item_impl(key, value)
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/basecontainer.py", line 549, in _set_item_impl
    self._validate_set(key, value)
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/dictconfig.py", line 180, in _validate_set
    target = self._get_node(key) if key

Let's now train a centered-instance model.

In [ ]:
!sleap-nn train --config baseline.centered_instance.yaml "data_config.train_labels_path=[dataset/drosophila-melanogaster-courtship/courtship_labels.slp]" trainer_config.ckpt_dir="models" trainer_config.run_name="courtship.topdown_confmaps" trainer_config.max_epochs=200

2026-07-02 08:48:06 | Input config:
2026-07-02 08:48:06 | 
data_config:
  train_labels_path:
  - dataset/drosophila-melanogaster-courtship/courtship_labels.slp
  val_labels_path: null
  validation_fraction: 0.1
  use_same_data_for_val: false
  test_file_path: null
  provider: LabelsReader
  user_instances_only: true
  data_pipeline_fw: torch_dataset
  cache_img_path: null
  use_existing_imgs: false
  delete_cache_imgs_after_training: true
  preprocessing:
    ensure_rgb: false
    ensure_grayscale: false
    max_height: null
    max_width: null
    scale: 1.0
    crop_size: null
    min_crop_size: 100
    crop_padding: null
  use_augmentations_train: true
  augmentation_config:
    intensity:
      uniform_noise_min: 0.0
      uniform_noise_max: 0.04
      uniform_noise_p: 0.0
      gaussian_noise_mean: 0.0
      gaussian_noise_std: 0.02
      gaussian_noise_p: 0.0
      contrast_min: 0.9
      contrast_max: 1.1
      contrast_p: 0.0
      brightness_min: 0.9
      brightness_max: 1.1


The models (along with the profiles and ground truth data used to train and validate the model) are saved in the `models/` directory:

In [ ]:
!tree models/

models/
├── courtship.centroid
│   ├── best.ckpt
│   ├── initial_config.yaml
│   ├── labels_train_gt_0.slp
│   ├── labels_val_gt_0.slp
│   ├── training_config.yaml
│   └── training_log.csv
└── courtship.topdown_confmaps
    ├── best.ckpt
    ├── initial_config.yaml
    ├── labels_train_gt_0.slp
    ├── labels_val_gt_0.slp
    ├── training_config.yaml
    └── training_log.csv

3 directories, 12 files


## Inference
Let's run inference with our trained models for centroids and centered instances.

In [ ]:
!sleap-nn track -i "dataset/drosophila-melanogaster-courtship/20190128_113421.mp4" --frames 0-100 -m "models/courtship.centroid" -m "models/courtship.topdown_confmaps"

2025-09-24 00:12:47 | INFO | sleap_nn.predict:run_inference:319 | Started inference at: 2025-09-24 00:12:47.871203
2025-09-24 00:12:47 | INFO | sleap_nn.predict:run_inference:330 | Integral refinement is not supported with MPS device. Using CPU.
2025-09-24 00:12:47 | INFO | sleap_nn.predict:run_inference:335 | Using device: cpu
Predicting... ━━━━━━━━━━━━━━━ 100% 101/101 ETA: 0:00:00 Elapsed: 0:00:17 6.0 FPS0 FPS7 FPS
2025-09-24 00:13:05 | INFO | sleap_nn.predict:run_inference:453 | Finished inference at: 2025-09-24 00:13:05.531684
2025-09-24 00:13:05 | INFO | sleap_nn.predict:run_inference:454 | Total runtime: 17.660489082336426 secs
2025-09-24 00:13:05 | INFO | sleap_nn.predict:run_inference:465 | Predictions output path: dataset/drosophila-melanogaster-courtship/20190128_113421.predictions.slp
2025-09-24 00:13:05 | INFO | sleap_nn.predict:run_inference:466 | Saved file at: 2025-09-24 00:13:05.594857


When inference is finished, predictions are saved in a file. Since we didn't specify a path, it will be saved as `<video filename>.predictions.slp` in the same directory as the video:

In [ ]:
!tree dataset/drosophila-melanogaster-courtship

dataset/drosophila-melanogaster-courtship
├── 20190128_113421.mp4
├── 20190128_113421.mp4.predictions.slp
├── 20190128_113421.predictions.slp
├── courtship_labels.slp
└── example.jpg

1 directory, 5 files


If you're using Chrome you can download your trained models like so:

In [ ]:
# Zip up the models directory
!zip -r trained_models.zip models/

# Download.
from google.colab import files
files.download("/content/trained_models.zip")

  adding: models/ (stored 0%)
  adding: models/courtship.topdown_confmaps/ (stored 0%)
  adding: models/courtship.topdown_confmaps/labels_pr.val.slp (deflated 74%)
  adding: models/courtship.topdown_confmaps/metrics.val.npz (deflated 0%)
  adding: models/courtship.topdown_confmaps/labels_pr.train.slp (deflated 67%)
  adding: models/courtship.topdown_confmaps/labels_gt.val.slp (deflated 72%)
  adding: models/courtship.topdown_confmaps/initial_config.json (deflated 73%)
  adding: models/courtship.topdown_confmaps/training_log.csv (deflated 55%)
  adding: models/courtship.topdown_confmaps/metrics.train.npz (deflated 0%)
  adding: models/courtship.topdown_confmaps/labels_gt.train.slp (deflated 61%)
  adding: models/courtship.topdown_confmaps/best_model.h5 (deflated 8%)
  adding: models/courtship.topdown_confmaps/training_config.json (deflated 88%)
  adding: models/courtship.centroid/ (stored 0%)
  adding: models/courtship.centroid/labels_pr.val.slp (deflated 82%)
  adding: models/courtship

And you can likewise download your predictions:

In [ ]:
from google.colab import files
files.download('dataset/drosophila-melanogaster-courtship/20190128_113421.mp4.predictions.slp')

In some other browsers (Safari) you might get an error and you can instead download using the "Files" tab in the side panel (it has a folder icon). Select "Show table of contents" in the "View" menu if you don't see the side panel.